# Principles of MRI Lab: Image Formation

This lab explores MRI image formation, including slice selection and frequency encoding.  These use modules from the Resonint ilumr-courseware, Labs 2 and 3.

>-------------------------------------------------------------------------------------------------------------------------------------------------------
> #### **Setup Task: Run the Notebook**
> 
> 1. Edit the cell below to set the `LAB_USER_NAME` variable to your name
> 2. Click **Run->Run All Cells** in the in top menu bar of jupyterlab
> 3. Open the Table of Contents side-bar on the left edge of jupyterlab to aid in navigation
> 
> -------------------------------------------------------------------------------------------------------------------------------------------------------

In [ ]:
LAB_USER_NAME = 'Student'

**Important**: To initialise this notebook, edit the cell above to set `LAB_USER_NAME` to your name, then click **Run->Run All Cells** in the top menu bar.

In [ ]:
import panel as pn
import sys
import os
import yaml
import numpy as np

from matipo import Sequence, SEQUENCE_DIR, GLOBALS_DIR, DATA_DIR, Unit
from matipo.util.decimation import decimate
from matipo.util.fft import fft_reconstruction
from matipo.util.etl import deinterlace

from matipo.experiment.base_experiment import BaseExperiment, auto_inputs, PlotInterface
from matipo.experiment.models import PLOT_COLORS, SITickFormatter, SIHoverFormatter

from matipo.experiment.plots import SignalPlot, SpectrumPlot, ImagePlot, Image1DPlot, ComplexDataLinePlot

pn.extension(inline=True)

WORKSPACE = os.path.join(LAB_USER_NAME)
LAB_DIR = os.path.join('/home/data/PrinciplesOfMRI', LAB_USER_NAME)
os.makedirs(LAB_DIR, exist_ok=True)
print('Data will be saved to', LAB_DIR)

GAMMA_BAR = 42.58e6
RESOLUTION = 64
FOV = 12e-3 # m
AXIS_VECTORS = {
    'X': np.array([1,0,0], dtype=float),
    'Y': np.array([0,1,0], dtype=float),
    'Z': np.array([0,0,1], dtype=float)
}

try:
    with open(GLOBALS_DIR+'hardpulse_90.yaml', 'r') as f:
        data = yaml.load(f, Loader=yaml.SafeLoader)
        AREA_90 = data['a_90']*data['t_90']
except IOError:
    print('Unable to load pulse calibration, using default value')
    AREA_90 = 0.3*32e-6
    
def estimate_a_90(pulse_shape, pulse_width):
    shape_area = np.mean(pulse_shape)*pulse_width
    return min(1, abs(AREA_90/shape_area))

## Radiofrequency Pulses and Slice Selection

In order to image a single slice rather than the entire sample, two main changes need to be made to the pulse sequence:

1) A slice-select gradient must be added along the axis perpendicular to the plane of the desired slice. This causes the spins along the chosen axis to precess at different frequencies making it possible to selectively excite them by transmitting at a specific frequency.

2) The RF excitation pulse must be modified to only transmit a limited range of frequencies. Prior experiments have used short (~20 us) high amplitude pulses which excite a very broad range of frequencies (called a "hard pulse"). For selective excitation, sinc-like RF pulse shapes are used, as they contain a limited range of frequencies (the "bandwidth").  The bandwidth of the RF pulse is inversely proportional to its duration, so longer pulses excite a narrower range of frequencies.


<center><img src="Images/slice.png" width="1000"></center>
<center><figcaption style="width: 600px;">Slicing along the z axis allows the sample to be imaged at a specific cross section index     </figcaption></center>


The relationship of the slice thickness ($\Delta z$) to RF pulse bandwidth ($BW_{rf}$) and slice-select gradient strength ($G_{Z,SS}$) is governed by the following equation:

$$\Delta z = \frac{BW_{rf}}{\bar\gamma \cdot G_{Z,SS}}$$

<center><img src="Images/slice_thickness.png" width="800"></center>
<center><figcaption style="width: 800px;">How the slice thickness is related to the RF pulse bandwidth and slice-select gradient strength </figcaption></center>

For small tip angles ($\alpha \ll 90^\circ$), the small-tip-angle approximation states that the RF pulse profile is approximately proportional to the spectrum, or Fourier Transform, of the RF pulse shape. If the tip angle at any point in the sample approaches or exceeds $90^\circ$ this approximation breaks down and the signal spectrum will be distorted.

For slice-selection, the goal is to excite the magnetization equally across a region in space, and without excitation outside of that region.  In other words, the goal is to create a sharp boundary between the excited and non-excited regions. This can be achieved when the frequency spectrum of the RF pulse has a rectangular shape, which is achieved with approximately sinc-shaped RF pulse shapes as per the small-tip-angle approximation:

<center><img src="Images/soft_pulse_parameters.png" width="600"></center>
<center><figcaption style="width: 600px;">An ideal RF pulse would create a rectangular slice select profile</figcaption></center>

In practice, only a truncated Sinc function is feasible.  With this truncation, additional windowing (e.g. Gaussian) is often applied to reduce ringing in the RF pulse profile. This process is called Apodization.

> -------------------------------------------------------------------------------------------------------------------------------------------------------
> #### **Visualising Sinc RF Pulse Design**
>The experiment below is designed to help you visualise how changing the parameters of a soft pulse changes the profile in the frequency domain. The RF pulse you design will be used for the following experiments.
> 1. Experiment with the three soft pulse parameters and observe how the shape of the RF-pulse and slice profile changes. 
> 2. How does changing the lobe width affect the spectral bandwidth?
> 3. How does adding more lobes improve the shape of the slice profile? What is the trade-off regarding the overall pulse width?
> 4. What happens when apodization is too large? Pick a value that looks like a good compromise.
> -------------------------------------------------------------------------------------------------------------------------------------------------------

In [ ]:
from custom_RF import CustomRF

class SincRF(CustomRF):
    
    def setup(self):
        super().setup()
        self.title = "Sinc RF Pulse Design"
        self.workspace = WORKSPACE
        
        self.inputs = {
            'lobe_width': pn.widgets.FloatInput(name="Lobe Width (μs)", start=50, end=400, step=50, value=100, width=80),
            'n_lobe': pn.widgets.FloatInput(name="Lobes", start=0, end=10, step=1, value=3, width=80),
            'apodization': pn.widgets.FloatInput(name="Apodization", start=0, end=100, step=1, value=0, width=80)
        }
        
        self.inputs['tbw'] = pn.widgets.FloatInput(name='Time Bandwidth Product', start=0, end=20, value=6, width=80, disabled=True)
        self.inputs['Trf'] = pn.widgets.FloatInput(name='RF Pulse Duration (μs)', start=0, end=4000, value=300, width=80, disabled=True)

        def sincRF_read_params(*events):
            self.inputs['tbw'].value = 2.0 * self.inputs['n_lobe'].value
            self.inputs['Trf'].value = self.inputs['lobe_width'].value * 2.0 * self.inputs['n_lobe'].value

        self.inputs['lobe_width'].param.watch(sincRF_read_params, ['value'], onlychanged=True)
        self.inputs['n_lobe'].param.watch(sincRF_read_params, ['value'], onlychanged=True)


        # auto frequency range based on max bandwidth
        BW_max = 1e-6*self.inputs['lobe_width'].start
        self.plots['spectrum'].fig.x_range.start = -4/BW_max
        self.plots['spectrum'].fig.x_range.end = 4/BW_max
        
        # update plots when any value is changed
        self.inputs['lobe_width'].param.watch(self.pulse_settings_handler, 'value_throttled')
        self.inputs['n_lobe'].param.watch(self.pulse_settings_handler, 'value_throttled')
        self.inputs['apodization'].param.watch(self.pulse_settings_handler, 'value_throttled')

        # manually trigger for first update
        self.inputs['lobe_width'].param.trigger('value_throttled')
    
    
    def calc_pulse_shape(self):
        width = max(1, self.inputs['n_lobe'].value)*self.inputs['lobe_width'].value*1e-6 # max(1, ...) to prevent width=0 when n_lobes=0
        N = int(width / 1e-6)
        if N > 1000:
            N = 1000
        dt = width / N
        pts = np.sinc(np.linspace(-self.inputs['n_lobe'].value, self.inputs['n_lobe'].value,N))
        pts*=np.exp(-self.inputs['apodization'].value*(np.linspace(-1, 1, N)**2))
        return width, pts
        

VisualizeRFExperiment = SincRF(state_id='lab3')
VisualizeRFExperiment()

> -------------------------------------------------------------------------------------------------------------------------------------------------------
> #### **1D Slice Profile**
> With the Sinc pulse designed, we can now use it to create a 1D image of the slice profile.
> 1. **Question:** Based on the bandwidth ($BW_{rf}$) of the Sinc pulse you have designed and a slice select gradient $G_{ss}=100 \ \mathrm{mT}/\mathrm{m}$, calculate the expected thickness of the slice profile.
> 2. Insert the **Shim Sample** (make sure to centre the sample using the depth gauge), set the slice select gradient, and run the experiment.
> 3. **Question:** How does the width of the slice profile compare to your predicted width?
> 4. Try changing the flip angle to 30 and 180 and compare the resulting spectrum.
> 5. **Question:** At what flip angle (out of 30, 90, or 180) does the signal spectrum look most similar to the RF pulse spectrum shown above?
> 1. Try changing the frequency offset of the RF pulse.
> 1. **Question:** How does the slice profile change when the frequency offset is changed?
> -------------------------------------------------------------------------------------------------------------------------------------------------------

In [ ]:
class SliceProfile(BaseExperiment):
    
    def setup(self):
        self.title = "Sinc RF Pulse Slice Profile Measurement"
        self.workspace = WORKSPACE
        self.seq = Sequence('custom_pulse_SE.py')
        self.plots = {
            'signal': SignalPlot(),
            'image': Image1DPlot()
        }
        
        with open(os.path.join(GLOBALS_DIR, 'gradient_calibration.yaml'), 'r') as f:
            data = yaml.load(f, Loader=yaml.SafeLoader)
            self.G_CAL = 1/(GAMMA_BAR*data['gradient_calibration']) # convert
            self.log.debug(f'G_CAL: {self.G_CAL}')
        
        self.inputs={
            'freq_offset':pn.widgets.FloatInput(name="RF Pulse Frequency Offset (kHz)", start=-30, end=30, step=1, value=0, width=200, disabled=False),
            'GSS':pn.widgets.FloatInput(name="Slice Select Gradient (mT/m)", start=0, end=150, step=10, value=100, width=200, disabled=False),
            'flip_angle':pn.widgets.FloatInput(name="Target Flip Angle (degrees)", start=10, end=360, step=10, value=90, width=200, disabled=False)
        }
        self.DEC = 4
        self.G_IMAGE = 0.1 # T/m
    
    def update_par(self):
        self.seq.setpar(
            a_90=self.inputs['flip_angle'].value/90*estimate_a_90(VisualizeRFExperiment.pulse_shape,VisualizeRFExperiment.pulse_width),
            t_90=VisualizeRFExperiment.pulse_width,
            shape_90=VisualizeRFExperiment.pulse_shape,
            f_slice_offset=1e3*self.inputs['freq_offset'].value,
            g_slice=(0,0,-1e-3*self.inputs['GSS'].value/self.G_CAL),
            g_read=(0,0,self.G_IMAGE/self.G_CAL),
            n_scans=4,
            n_samples=200*self.DEC,
            t_dw=20e-6/self.DEC, # using a long dwell time for narrow bandwith to more easily see the spectrum shape
            t_end=0.5
        )
    
    async def update_plots(self, *args, **kwargs):
        await self.seq.fetch_data()
        self.plots['signal'].update(seqdata=self.seq.data, t_dw=self.seq.par.t_dw)
        self.plots['image'].update(
            seqdata=self.seq.data,
            t_dw=self.seq.par.t_dw,
            g_read_mag=self.G_CAL*np.linalg.norm(self.seq.par.g_read),
            dec=self.DEC
        )
            

SliceProfileExperiment = SliceProfile(state_id='sinc_slice_profile')
SliceProfileExperiment()

## Frequency Encoding

<figure style="float: right;">
<img src="Images/GradCoilZ.png" width="400">
<figcaption style="width: 300px;">Z gradient coils and current directions (orange arrows) in the ilumr system. At $z_1$ the total field strength is increased, and at $z_2$ it is decreased.  Note that this design is different from most horizontal bore MRI systems due to the different orientation of the main magnetic field.</figcaption>
</figure>

The fundamental principle of image formation in MRI is to create controlled variations in the magnetic field, thereby causing different locations to resonate at different frequencies.  This is done with combinations of coils that create linear vartiations, or **gradients**, in the magnetic field along the $x$, $y$, and $z$ axes.

Because the gradients are linear, the Larmor frequency at any location will be a linear function of its coordinate along the gradient axis, e.g. when applying gradient fields with amplitudes $\vec G = [G_x, G_y, G_z]$:

$$ f = f_0 + \bar\gamma\ \vec G \cdot \vec r$$

Where $f_0$ is the Larmor frequency when no gradient is applied, and $\bar\gamma$ is the gyromagnetic ratio ($\bar\gamma = 42.58 \times 10^{6} \ \mathrm{Hz}/\mathrm{T}$).

<figure style="float: right;">
<img src="Images/WaterSampleAddition.png" width="500">
<center><figcaption style="width: 500px;">Frequency Encoding</figcaption><center>
</figure>

An example of frequency encoding is shown in the figure at the right.  The signal from each partition will have a different frequency depending on its position, but the measured signal will be the sum of all of them, a complex shape caused by constructive and destructive intereference. The frequencies present in the measured signal can be extracted using a Fourier Transform, revealing individual peaks in the frequency domain associated with the partitions.

In this example the partitions of water are separated, but the same logic can be applied to a continuous shape. We can imagine slicing the sample into multiple slices along the gradient axis, so that the signals from the slices are similar to the signals from the partitions. When the total signal is measured and processed with a Fourier transform, the magnitude of the resulting spectrum at any frequency will be proportional to the amount of sample in the slice that is precessing at that frequency. This means the spectrum is a 1D projection image of the sample.

Mathematically, the total signal $s(t)$ we measure in the rotating frame is proportional to the integral of the magnetization $m(x)$ multiplied by a phasor contributed by the frequency shift caused by the gradient.  Assuming a constant frequency encoding gradient in the $x$-direction, $\vec G (t) = [G_{xr},0,0]$:

$$ \large s(t) \propto \int m(x) e^{-i \gamma \ G_{xr} x t} dx$$

With the substitution $k_x = \bar\gamma G_{xr} t$, this equation has the same form as the Fourier transform:

$$ \large s(t) \propto \int m(x) e^{-i 2 \pi k_x x} dx = \mathcal{F}\{m(x)\}$$

And the object magnetization can be recovered by taking the inverse Fourier transform of the measured signal:

$$ \large \hat{m}(x) = \mathcal{F}^{-1}\{s(t)\}$$

### Pulse Sequence

<figure style="float: right;">
<img src="Images/se_2.png" width="500">
<figcaption style="width: 500px;">A Spin Echo sequence with a frequency encoding gradient along the $z$-axis ($G_z$) and gated acquisition.</figcaption>
</figure>

To create frequency encoding, the magnetic field gradient is turned on during the acquisition.  
 The gradient also does not need to be active for the entire pulse sequence, only during the acquisition time and for a short time between the $90^\circ$ and $180^\circ$ RF pulses – to satisfy a requirement for optimal spin echo formation where the area under the gradient between the $90^\circ$ and $180^\circ$ pulses must match the area under the gradient from the $180^\circ$ to the echo time ($T_E$). 
 
 This pulse sequence will be used in the next experiment, with the modification that the frequency encoding gradient is applied on any combination of the gradients.  This modification will allow for creation of projection images in the x, y, and z directions, as well as any combination of the three.  The resulting images will be 1D projections of the sample along the gradient direction.

>---
>#### **Frequency Encoding**
>1. Insert a phantom at the correct depth and start the experiment below with **"Run Loop"**.
>2. Adjust one of the read gradients to 25%.  
>1. **Question:** What do you see in the spectrum?  Does this match your expectations based on the phantom you inserted?
>1. Trying increasing and decreasing the read gradient strength.
>1. **Question:** How does the spectrum change when the read gradient is changed?
>1. Switch between using the X, Y, and Z read gradients.
>1. **Question:** How does the spectrum change?  What does this correspond to in the phantom?
>1. Repeat the experiment with a different phantom and note how the spectrum changes with manipulation of the read gradient strength and direction.
>---

In [ ]:
class FrequencyEncoding(BaseExperiment):
    
    def setup(self):
        self.enable_runloop = True
        self.title = "Frequency Encoding Experiment"
        self.workspace = WORKSPACE
        self.seq = Sequence('SE.py')
        self.plots = {
            'signal': SignalPlot(),
            'spectrum': SpectrumPlot(show_magnitude=True)
        }
        
        self.inputs = auto_inputs(self.seq, {
        })
        
        self.inputs['read_x_grad'] = pn.widgets.IntInput(name='Read Gradient X Component (%)', start=-100, end=100, value=0, width=200, disabled=False)
        self.inputs['read_y_grad'] = pn.widgets.IntInput(name='Read Gradient Y Component (%)', start=-100, end=100, value=0, width=200, disabled=False)
        self.inputs['read_z_grad'] = pn.widgets.IntInput(name='Read Gradient Z Component (%)', start=-100, end=100, value=0, width=200, disabled=False)
                
        self.DEC = 4
    
    def update_par(self):
        self.seq.setpar(
            g_read=(0.01*self.inputs['read_x_grad'].value, 0.01*self.inputs['read_y_grad'].value, 0.01*self.inputs['read_z_grad'].value),
            t_echo=0.012,
            n_scans=2,
            n_samples=100,
            t_dw=20e-6, # using a long dwell time for narrow bandwith to more easily see the spectrum shape
            t_end=0.1
        )
        
        # supersample for postprocess decimation
        self.seq.setpar(
            n_samples=self.seq.par.n_samples*self.DEC,
            t_dw=self.seq.par.t_dw/self.DEC
        )
    
    async def update_plots(self, *args, **kwargs):
        await self.seq.fetch_data()
        self.plots['signal'].update(seqdata=self.seq.data, t_dw=self.seq.par.t_dw)
        self.plots['spectrum'].update(seqdata=decimate(self.seq.data, self.DEC), t_dw=self.seq.par.t_dw, t0='auto', fast=True)
            

FrequencyEncodingExperiment = FrequencyEncoding(state_id='fe_experiment')
FrequencyEncodingExperiment()